In [13]:
import plotly.graph_objects as go

cat_to_feat = {
    "HLA Binding & Processing": [
        "mutant_rank_netMHCpan", "mutant_rank_PRIME", "mut_Rank_Stab",
        "mut_Thalf", "TAP_score", "mut_netchop_score_ct"
    ],
    "Differential Agretopicity": [
        "DAI_NetMHC", "DAI_MixMHC", "DAI_NetStab", "DAI_MixMHC_mbp"
    ],
    "Tumor Abundance & Expr.": [
        "GTEx_expression_mean", "TCGA_Cancer_expression", "Sample_Tissue_expression"
    ],
    "TCR Recognition Potential": [
        "Rank (pMTnet)", "foreignness_score", "hydrophobicity_avg_diff",
        "charge_tcr_contact", "hydroph_tcr_contact", "mw_tcr_contact"
    ]
}

feat_to_tool = {
    "mutant_rank_netMHCpan": "NetMHCpan 4.1", "mutant_rank_PRIME": "PRIME",
    "mut_Rank_Stab": "NetMHCstabpan", "mut_Thalf": "NetMHCstabpan",
    "TAP_score": "NetMHCpan 4.1", "mut_netchop_score_ct": "NetChop",
    "DAI_NetMHC": "NetMHCpan 4.1", "DAI_MixMHC": "MixMHCpred",
    "DAI_NetStab": "NetMHCstabpan", "DAI_MixMHC_mbp": "MixMHCpred",
    "GTEx_expression_mean": "GTEx Database", "TCGA_Cancer_expression": "TCGA Database",
    "Sample_Tissue_expression": "GTEx Database",
    "Rank (pMTnet)": "pMTnet", "foreignness_score": "Biopython / Alignment",
    "hydrophobicity_avg_diff": "PyChem / Biopython", "charge_tcr_contact": "PyChem / Biopython",
    "hydroph_tcr_contact": "PyChem / Biopython", "mw_tcr_contact": "PyChem / Biopython"
}


In [14]:
categories = list(cat_to_feat.keys())
features = []
for cat in categories: features.extend(cat_to_feat[cat])

# Ordenamos las herramientas para que coincidan verticalmente con las categorías
# HLA arriba, TCR abajo.
tools_ordered = [
    "NetChop", "PRIME", "NetMHCpan 4.1", "NetMHCstabpan", "MixMHCpred", # Tools HLA/DAI
    "GTEx Database", "TCGA Database",                                   # Tools Tumor
    "pMTnet", "Biopython / Alignment", "PyChem / Biopython"             # Tools TCR
]

all_nodes = categories + features + tools_ordered
node_map = {name: i for i, name in enumerate(all_nodes)}

cat_colors_map = {
    "HLA Binding & Processing": "#1f77b4", "Differential Agretopicity": "#ff7f0e",
    "Tumor Abundance & Expr.": "#d62728", "TCR Recognition Potential": "#2ca02c"
}


In [15]:
source_indices = []
target_indices = []
values = []
link_colors = []

for cat in categories:
    cat_idx = node_map[cat]
    base_hex = cat_colors_map[cat]
    # Colores con transparencia
    color_step1 = f"rgba{tuple(int(base_hex.lstrip('#')[j:j+2], 16) for j in (0, 2, 4)) + (0.4,)}"
    color_step2 = f"rgba{tuple(int(base_hex.lstrip('#')[j:j+2], 16) for j in (0, 2, 4)) + (0.2,)}"
    
    for feat in cat_to_feat[cat]:
        feat_idx = node_map[feat]
        # Link 1
        source_indices.append(cat_idx); target_indices.append(feat_idx); values.append(1)
        link_colors.append(color_step1)
        # Link 2
        tool_name = feat_to_tool[feat]
        tool_idx = node_map[tool_name]
        source_indices.append(feat_idx); target_indices.append(tool_idx); values.append(1)
        link_colors.append(color_step2)

# Colores de Nodos (Categorías color, resto Gris Oscuro sólido)
node_colors = [cat_colors_map[n] if n in categories else "#333333" for n in all_nodes]


In [16]:
fig = go.Figure(data=[go.Sankey(
    arrangement = "snap",
    node = dict(
        pad = 20, thickness = 15,
        line = dict(color = "black", width = 0.5),
        label = all_nodes,
        color = node_colors
    ),
    link = dict(
        source = source_indices, target = target_indices,
        value = values, color = link_colors
    ),
    # FUERZA EL TEXTO DE LOS NODOS A NEGRO
    textfont = dict(color = "black", size = 12, family = "Arial") 
)])

fig.update_layout(
    title=dict(
        text="<b>Data Provenance: Feature Engineering Architecture</b>",
        font=dict(
            size=22, 
            color="black",  # <--- ESTO FUERZA EL NEGRO EN EL TÍTULO
            family="Arial"
        ),
        x=0.5,
        xanchor='center'
    ),
    font=dict(color="black"), # Fuerza negro en todo lo demás por si acaso
    width=1300,
    height=800,
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(t=80, l=20, r=20, b=20)
)

fig.write_html("sankey_professional_final.html")
print("✅ Diagrama guardado. Texto negro, sin líneas grises y con cruces minimizados.")
fig.show()

✅ Diagrama guardado. Texto negro, sin líneas grises y con cruces minimizados.
